In [1]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick brown dog jumps over the lazy fox",
    "the cat sits on the mat",
    "the dog sits on the rug",
    "machine learning learns patterns from data",
    "deep learning models learn patterns",
    "word embeddings represent words as vectors",
    "similar words occur in similar contexts",
    "the model learns word context",
    "machine learning uses data and patterns",
    "deep learning uses neural networks",
    "the fox jumps over the dog",
    "the cat and dog are animals",
    "word context helps predict words",
    "neural networks learn useful features"
]

tokens = [re.findall(r"[a-z]+", s.lower()) for s in corpus]
vocab = sorted(set(w for sentence in tokens for w in sentence))

word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for w, i in word2id.items()}

print("Vocabulary size:", len(vocab))
print("Sample tokens:", tokens[0])

Vocabulary size: 44
Sample tokens: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']


In [2]:
window = 2
contexts, targets = [], []

for sentence in tokens:
    for i in range(window, len(sentence) - window):
        context = sentence[i-2:i] + sentence[i+1:i+3]
        contexts.append([word2id[w] for w in context])
        targets.append(word2id[sentence[i]])

X = np.array(contexts)
y = np.array(targets)

print("Training examples:", len(X))
print("Context words:", [id2word[i] for i in X[0]])
print("Target word:", id2word[y[0]])

Training examples: 31
Context words: ['the', 'quick', 'fox', 'jumps']
Target word: brown


In [5]:
model = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Embedding(len(vocab), 16),
    layers.GlobalAveragePooling1D(),
    layers.Dense(len(vocab), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(X, y, epochs=200, batch_size=4, verbose=0)

print("Model training completed!")
print("Final accuracy:", round(history.history["accuracy"][-1], 3))

Model training completed!
Final accuracy: 0.935


In [6]:
i = 0
probabilities = model.predict(X[i:i+1], verbose=0)[0]
predicted = np.argmax(probabilities)

print("Context:", [id2word[int(w)] for w in X[i]])
print("Actual target:", id2word[y[i]])
print("Predicted target:", id2word[predicted])

print("\nTop predictions:")
for j in np.argsort(probabilities)[-3:][::-1]:
    print(id2word[j], round(float(probabilities[j]), 3))

Context: ['the', 'quick', 'fox', 'jumps']
Actual target: brown
Predicted target: brown

Top predictions:
brown 0.514
over 0.22
the 0.071
